# Chronos-2: Zero-Shot Inference Demo

Chronos-2 is Amazon's zero-shot probabilistic time series foundation model — a decoder-only transformer trained on 100B+ synthetic and real data points. **Zero-shot** means no fine-tuning is needed: the model generalises directly to new time series it has never seen.

This notebook walks through basic inference on our S&P 500 dataset using the local GPU:

1. Install / verify dependencies
2. Load AAPL close prices from `data/sp500_macro_master.csv`
3. Load `amazon/chronos-2` via `Chronos2Pipeline`
4. Run a 5-step-ahead probabilistic forecast
5. Visualise the 80% prediction interval

In [ ]:
import importlib
import subprocess
import sys

packages = {
    'chronos': 'git+https://github.com/amazon-science/chronos-forecasting.git',
    'torch': 'torch',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
}

for pkg, install_name in packages.items():
    try:
        importlib.import_module(pkg)
        print(f'{pkg}: already installed')
    except ImportError:
        print(f'{pkg}: installing...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', install_name])
        print(f'{pkg}: done')

## 1. Library Imports & Device Setup

**API note — `Chronos2Pipeline` vs `ChronosPipeline`:**

| Class | Model family | Architecture |
|---|---|---|
| `ChronosPipeline` | `chronos-t5-*` (v1) | Encoder–decoder (T5) |
| `Chronos2Pipeline` | `amazon/chronos-2` (v2) | Decoder-only transformer |

Using `ChronosPipeline` with `amazon/chronos-2` will raise an error. Always import and use `Chronos2Pipeline` for the v2 model.

**bfloat16 (Brain Float 16):** same exponent range as float32 but half the memory footprint. Ideal for GPU inference — negligible accuracy impact, ~50% VRAM saving.

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from chronos import Chronos2Pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 2. Data Loading

`sp500_macro_master.csv` contains 99 S&P 500 tickers with 5 years of daily OHLCV data and 11 macro covariates (FRED series: DGS10, UNRATE, CPIAUCSL, etc.).

We filter for **AAPL** and take the last **252 trading days (~1 year)** as the context window — the historical slice the model conditions on when producing the forecast.

In [ ]:
DATA_PATH = 'data/sp500_macro_master.csv'
TICKER    = 'AAPL'
CONTEXT_LENGTH = 252

df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df = df[df['Ticker'] == TICKER].sort_values('Date').reset_index(drop=True)
context_df = df.tail(CONTEXT_LENGTH).copy()
context    = torch.tensor(context_df['Close'].values, dtype=torch.float32)
dates      = pd.to_datetime(context_df['Date'].values)

print(f'Ticker:  {TICKER}')
print(f'Context: {CONTEXT_LENGTH} days ({dates[0].date()} to {dates[-1].date()})')
print(f'Close range: ${float(context.min()):.2f} to ${float(context.max()):.2f}')
print(f'Last close:  ${float(context[-1]):.2f}')
print(f'Tensor: shape={context.shape}  dtype={context.dtype}')

## 3. Model Loading

`Chronos2Pipeline.from_pretrained('amazon/chronos-2')` downloads the model weights (~400 MB) on first run and caches them locally in `~/.cache/huggingface/hub`. Subsequent runs load from cache instantly.

`torch_dtype=torch.bfloat16` reduces VRAM usage by ~50% vs float32 with negligible accuracy impact for inference — BF16 preserves the same exponent range as FP32.

In [ ]:
print('Loading amazon/chronos-2 on', device, '...')
pipeline = Chronos2Pipeline.from_pretrained(
    'amazon/chronos-2',
    device_map='cuda',
    torch_dtype=torch.bfloat16,
)
print('Model loaded successfully.')
if torch.cuda.is_available():
    used_mb = torch.cuda.memory_allocated() / 1024**2
    print(f'VRAM allocated: {used_mb:.0f} MB')

## 4. Basic Inference — 5-Step Ahead Forecast

`prediction_length=5` = 5 trading days (~1 week ahead).

Chronos-2 internally samples multiple trajectories and returns a tensor of quantile forecasts.

**Output shape:** `(1, 9, 5)` = `(n_variates, n_quantile_levels, horizon)`

The 9 quantile levels are indexed 0–8:

| Index | Quantile |
|---|---|
| 0 | q10 |
| 4 | q50 (median) |
| 8 | q90 |

q10–q90 forms an **80% prediction interval**.

> **Note:** `limit_prediction_length=False` is required to allow `prediction_length` values below the model's default minimum.

In [ ]:
PREDICTION_LENGTH = 5

print(f'Running {PREDICTION_LENGTH}-step ahead forecast for {TICKER}...')
with torch.inference_mode():
    forecast = pipeline.predict(
        context=context,
        prediction_length=PREDICTION_LENGTH,
        limit_prediction_length=False,
    )

print(f'Output tensor shape: {forecast.shape}')
print(f'  (n_variates={forecast.shape[0]}, n_quantiles={forecast.shape[1]}, horizon={forecast.shape[2]})')

# Extract quantiles — 9 levels indexed 0..8
median = forecast[0, 4, :].cpu().numpy()   # q50
q10    = forecast[0, 0, :].cpu().numpy()   # q10
q90    = forecast[0, 8, :].cpu().numpy()   # q90

last_close = float(context[-1])
print(f'\nForecast — next {PREDICTION_LENGTH} trading days after {dates[-1].date()}:')
print(f'  {"Day":<6} {"Q10 ($)":>10} {"Median ($)":>12} {"Q90 ($)":>10}')
for i in range(PREDICTION_LENGTH):
    print(f'  {i+1:<6} {q10[i]:>10.2f} {median[i]:>12.2f} {q90[i]:>10.2f}')

exp_return = (median[-1] - last_close) / last_close * 100
ci_width   = q90[-1] - q10[-1]
print(f'\nExpected return (day 5 median):  {exp_return:+.2f}%')
print(f'80% CI width (day 5):            ${ci_width:.2f}')

## 5. Visualization

We plot the last **30 days** of historical Close prices for visual clarity (the full 252-day context was used for inference). The vertical dotted line marks the forecast boundary.

- **Black line** — historical close prices
- **Red dashed line** — point forecast (median, q50)
- **Shaded red band** — 80% prediction interval (q10–q90)

The forecast series is stitched to the last historical close to avoid a gap in the chart.

In [ ]:
PLOT_HISTORY = 30

hist_dates  = dates[-PLOT_HISTORY:]
hist_prices = context[-PLOT_HISTORY:].numpy()
last_date   = dates[-1]

# Forecast dates (business days only, skip weekends)
fcast_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=PREDICTION_LENGTH)

fig, ax = plt.subplots(figsize=(13, 5))

# Historical prices
ax.plot(hist_dates, hist_prices,
        color='black', linewidth=2.2, label='Historical Close (30d)', zorder=3)
ax.plot(last_date, float(context[-1]),
        'ko', markersize=7, zorder=5)

# Connect last historical to forecast — stitch seamlessly
conn_x = [last_date] + list(fcast_dates)
conn_median = [float(context[-1])] + list(median)
conn_q10    = [float(context[-1])] + list(q10)
conn_q90    = [float(context[-1])] + list(q90)

ax.plot(conn_x, conn_median,
        color='crimson', linewidth=2, linestyle='--',
        label='Median Forecast (q50)', zorder=3)
ax.fill_between(conn_x, conn_q10, conn_q90,
                color='crimson', alpha=0.15,
                label='80% Prediction Interval')
ax.scatter(fcast_dates, median, color='crimson', s=50, zorder=5)

# Vertical separator
ax.axvline(x=last_date, color='#888888', linestyle=':', linewidth=1.5,
           label='Forecast Start')

# Annotations: last close + day-5 median
ax.annotate(f'${float(context[-1]):.2f}',
            xy=(last_date, float(context[-1])),
            xytext=(-45, 10), textcoords='offset points',
            fontsize=9, color='black')
ax.annotate(f'${median[-1]:.2f}\n({exp_return:+.1f}%)',
            xy=(fcast_dates[-1], median[-1]),
            xytext=(5, 0), textcoords='offset points',
            fontsize=9, color='crimson')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
fig.autofmt_xdate(rotation=30)

ax.set_title(f'{TICKER}  ·  Chronos-2 Zero-Shot Forecast  ·  {PREDICTION_LENGTH}-Day Ahead  (80% CI)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()

import os
os.makedirs('notebooks', exist_ok=True)
plt.savefig('notebooks/01_chronos2_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: notebooks/01_chronos2_forecast.png')